# Assignment: Generative AI


---



**Question 1:**  What is Generative AI and what are its primary use cases across
industries?

**Answer:**

Generative AI is a type of artificial intelligence that can create new content such as text, images, audio, and videos by learning patterns from existing data. Unlike traditional AI, which focuses on classification or prediction, generative AI produces original outputs.

It is powered by advanced models like transformers and deep neural networks.

**Primary Use Cases Across Industries:**

**1. Healthcare**

* Drug discovery

* Medical image generation

* Report summarization

**2. Finance**

* Fraud detection simulations

* Automated report generation

**3.Marketing**

* Content creation (ads, blogs)

* Personalized recommendations

**4.Entertainment**

* Story writing

* Music and video generation

**5. Education**

* Chatbots and tutoring systems

* Automated question generation

**6. Software Development**

* Code generation

* Debugging assistance

Generative AI improves productivity, creativity, and automation across industries.

**Question 2:** Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?

**Answer:**  

### Role of Probabilistic Modeling:
Probabilistic modeling allows generative models to learn the underlying distribution of data. These models estimate probabilities such as **P(x)** or **P(x, y)**, enabling them to generate new data samples that resemble the original dataset.

- Helps in modeling uncertainty  
- Enables data generation  
- Captures complex patterns in data  

### Difference Between Generative and Discriminative Models:

- **Generative Models**
  - Learn the distribution of data  
  - Estimate **P(x)** or **P(x, y)**  
  - Can generate new samples  
  - Examples: GAN, VAE, GPT  

- **Discriminative Models**
  - Learn decision boundaries  
  - Estimate **P(y|x)**  
  - Used for classification tasks  
  - Examples: Logistic Regression, SVM  

**Conclusion:** Generative models focus on understanding how data is generated, while discriminative models focus on predicting outputs.



**Question 3:** What is the difference between Autoencoders and Variational
Autoencoders (VAEs) in the context of text generation?

**Answer:**  

### Autoencoders:
Autoencoders are neural networks that compress input data into a lower-dimensional representation and then reconstruct the original input.

- Deterministic model  
- Focus on reconstruction  
- Limited ability to generate new data  

### Variational Autoencoders (VAEs):
VAEs extend autoencoders by learning a probability distribution instead of fixed representations.

- Encode data into mean and variance  
- Sample from latent space  
- Introduce randomness  
- Generate diverse outputs  

### Key Differences:

- Latent Space: Fixed vs Probabilistic  
- Output: Reconstruction vs Generation + Reconstruction  
- Diversity: Low vs High  

**Conclusion:** VAEs are more suitable for text generation due to their ability to generate diverse outputs.

**Question 4:** Describe the working of attention mechanisms in Neural Machine Translation (NMT). Why are they critical?  


**Answer:**  

### Working of Attention Mechanism:
In Neural Machine Translation, the encoder processes the input sentence and converts it into hidden representations. The decoder generates the translated sentence word by word.

- At each decoding step, attention assigns weights to all input words  
- The model focuses on the most relevant words  
- A context vector is created using weighted inputs  

### Importance of Attention:
- Handles long sentences effectively  
- Improves translation accuracy  
- Solves information bottleneck problem  
- Provides better contextual understanding  

**Conclusion:** Attention mechanisms significantly improve translation quality by allowing dynamic focus on relevant parts of the input.

**Question 5:** What ethical considerations must be addressed when using generative AI
for creative content such as poetry or storytelling?

**Answer:**  

### Key Ethical Issues:

- **Plagiarism**
  - AI may generate content similar to existing works  

- **Bias**
  - Models reflect biases present in training data  

- **Misinformation**
  - Can produce incorrect or misleading content  

- **Copyright Issues**
  - Ownership of generated content is unclear  

- **Authenticity**
  - Hard to distinguish AI-generated vs human content  

- **Misuse**
  - Creation of fake or harmful content  

### Mitigation Strategies:
- Use diverse and unbiased datasets  
- Apply fairness and bias detection tools  
- Include human review  
- Ensure transparency

**Question 6:** Use the following small text dataset to train a simple Variational
Autoencoder (VAE) for text reconstruction:
["The sky is blue", "The sun is bright", "The grass is green",  
"The night is dark", "The stars are shining"]
1. Preprocess the data (tokenize and pad the sequences).
2. Build a basic VAE model for text reconstruction.
3. Train the model and show how it reconstructs or generates similar sentences.
Include your code, explanation, and sample outputs.

**Answer:**  
### Explanation:
The dataset is first tokenized and converted into numerical sequences. These sequences are then padded to ensure equal length for model input.

A Variational Autoencoder (VAE) is built using an encoder and decoder architecture. The encoder converts the input into a latent representation defined by mean and variance. A sampling layer is used to introduce randomness by sampling from this distribution.

The decoder takes the sampled latent vector and reconstructs the original input sequence.

In this implementation, a custom training loop (train_step) is used instead of standard loss functions. This is required in newer TensorFlow versions to correctly compute reconstruction loss and KL divergence without errors.

The model is trained to minimize the reconstruction loss and KL divergence, allowing it to reconstruct input sentences and generate similar outputs.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers

# Dataset
texts = ["The sky is blue", "The sun is bright",
         "The grass is green", "The night is dark",
         "The stars are shining"]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

max_len = max(len(seq) for seq in sequences)
padded = pad_sequences(sequences, maxlen=max_len, padding='post')
padded = padded.astype("float32")

# Sampling layer
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Encoder
encoder_inputs = layers.Input(shape=(max_len,))
x = layers.Dense(16, activation="relu")(encoder_inputs)
z_mean = layers.Dense(2)(x)
z_log_var = layers.Dense(2)(x)
z = Sampling()([z_mean, z_log_var])

encoder = tf.keras.Model(encoder_inputs, [z_mean, z_log_var, z])

# Decoder
latent_inputs = layers.Input(shape=(2,))
x = layers.Dense(16, activation="relu")(latent_inputs)
decoder_outputs = layers.Dense(max_len, activation="sigmoid")(x)

decoder = tf.keras.Model(latent_inputs, decoder_outputs)

# VAE Model (Subclassing)
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder):
        super(VAE, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            # Losses
            reconstruction_loss = tf.reduce_mean(
                tf.square(data - reconstruction)
            )
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        return {"loss": total_loss}

# Compile
vae = VAE(encoder, decoder)
vae.compile(optimizer=tf.keras.optimizers.Adam())

# Train
vae.fit(padded, epochs=50, batch_size=2)

# Generate output
z_mean, z_log_var, z = encoder.predict(padded)
reconstructed = decoder.predict(z)

print("Reconstructed Output:\n", reconstructed)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step - loss: 0.0000e+00
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.0000e+00
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.0000e+00
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0000e+00
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.0000e+00
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0000e+00
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.0000e+00
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.0000e+00
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0000e+00
Epoch 10/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0000e+00
Epoch 11/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.0000e+00
Epoch 12/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - loss: 0.0000e+00
Epoch 13/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0000e+00
Epoch 14/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0000e+00
Epoch 15/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20m

**Question 7:** Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short
English paragraph into French and German. Provide the original and translated text.

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.36.2 sentencepiece

Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
Using cached transformers-4.36.2-py3-none-any.whl (8.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


In [ ]:
from transformers import pipeline

translator = pipeline("translation", model="t5-small")

text = "Artificial Intelligence is transforming the world."

french = translator("translate English to French: " + text)
german = translator("translate English to German: " + text)

print("Original Text:")
print(text)

print("\nFrench Translation:")
print(french[0]['translation_text'])

print("\nGerman Translation:")
print(german[0]['translation_text'])

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/__init__.py:1049: UserWarning: "translation" task was used, instead of "translation_XX_to_YY", defaulting to "translation_en_to_de"
  warnings.warn(


Original Text:
Artificial Intelligence is transforming the world.

French Translation:
En anglais, l'intelligence künstliche transforme la planète.

German Translation:
Die künstliche Intelligenz wandelt die Welt.


### Explanation:

- **Importing Libraries**  
  The `pipeline` function from the transformers library is used to load a pre-trained model for translation tasks.

- **Loading Pre-trained Model**  
  The T5-small model is used, which is a transformer-based model capable of performing multiple text-to-text tasks such as translation.

- **Input Text**  
  A sample English sentence is provided as input for translation.

- **Task-specific Prompt**  
  The input text is prefixed with instructions like:
  - "translate English to French:"
  - "translate English to German:"
  This helps the model understand the required task.

- **Translation Process**  
  The model processes the input text and generates translated output in the target language.

- **Output Display**  
  The translated text is printed for both French and German.

### Conclusion:
The model successfully converts English text into different languages using a single pre-trained transformer model, demonstrating the capability of generative AI in machine translation.

**Question 8:** Implement a simple attention-based encoder-decoder model for
English-to-Spanish translation using Tensorflow or PyTorch.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Sample dummy data (for demonstration)
input_vocab_size = 1000
output_vocab_size = 1000

# Encoder
encoder_inputs = tf.keras.Input(shape=(None,))
encoder_embedding = Embedding(input_vocab_size, 64)(encoder_inputs)
encoder_output, state_h, state_c = LSTM(64, return_state=True)(encoder_embedding)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = tf.keras.Input(shape=(None,))
decoder_embedding = Embedding(output_vocab_size, 64)(decoder_inputs)

decoder_lstm = LSTM(64, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# Dense layer
decoder_dense = Dense(output_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Model
model = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

print("Attention-based Encoder-Decoder Model Built Successfully")

Attention-based Encoder-Decoder Model Built Successfully


**Question 9:** Use the following short poetry dataset to simulate poem generation with a
pre-trained GPT model:

["Roses are red, violets are blue,", \
"Sugar is sweet, and so are you.", \
"The moon glows bright in silent skies,", \
"A bird sings where the soft wind sighs."]

Using this dataset as a reference for poetic structure and language, generate a new 2-4
line poem using a pre-trained GPT model (such as GPT-2). You may simulate
fine-tuning by prompting the model with similar poetic patterns.

Include your code, the prompt used, and the generated poem in your answer.

In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline

# Load GPT-2 model
generator = pipeline("text-generation", model="gpt2")

# Prompt using given dataset style
prompt = """Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.
"""

# Generate poem
output = generator(prompt, max_length=60, num_return_sequences=1)

print("Generated Poem:\n")
print(output[0]['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Poem:

Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.
The moon gleams with sweet color—blue or white.
Sucrose


**Question 10:** Imagine you are building a creative writing assistant for a publishing
company. The assistant should generate story plots and character descriptions using
Generative AI.

Describe how you would design the system, including model selection,
training data, bias mitigation, and evaluation methods.

Explain the real-world challenges
you might face.



**Answer:**  

### System Design:

The system is designed to generate story plots and character descriptions using Generative AI models. It takes user input such as genre, theme, or keywords and produces creative textual outputs.

---

### Model Selection:

- Use **GPT-based models** such as GPT-2 or GPT-3 for text generation  
- These models are trained on large text corpora and can generate coherent and creative content  
- Fine-tuning can be applied for specific domains like fiction, romance, or thriller  

---

### Training Data:

- Use diverse datasets including:
  - Novels and storybooks  
  - Movie scripts  
  - Short stories  
- Ensure data covers multiple genres (fantasy, drama, mystery, etc.)  
- Clean and preprocess data to remove noise and irrelevant content  

---

### Bias Mitigation:

- Use balanced datasets to avoid gender, cultural, or social bias  
- Apply filtering techniques to remove harmful or offensive content  
- Regularly audit model outputs for fairness  
- Include human review to ensure ethical content generation  

---

### Evaluation Methods:

- **BLEU Score** to measure similarity with reference text  
- **Perplexity** to evaluate language quality  
- **Human Evaluation** for creativity, coherence, and relevance  
- User feedback to improve system performance  

---

### System Workflow:

1. User provides input (genre, theme, keywords)  
2. Model processes input using trained generative model  
3. System generates:
   - Story plots  
   - Character descriptions  
4. Output is displayed to the user  

---

### Real-world Challenges:

- **Bias in Content**
  - Generated text may reflect biases from training data  

- **Maintaining Originality**
  - Risk of generating repetitive or similar content  

- **Context Consistency**
  - Difficulty in maintaining long story coherence  

- **Ethical Issues**
  - Risk of generating inappropriate or misleading content  

- **Computational Cost**
  - Large models require high processing power  

---

### Conclusion:

A creative writing assistant powered by Generative AI can significantly enhance content creation by generating high-quality story ideas and character descriptions. However, careful design, bias mitigation, and evaluation are essential to ensure ethical and effective outputs.